# Adventure Works Internet Sales — Revised Analysis

**MSSP 6070 | Assignment 2**

This revision preserves the four original assignment questions while improving reproducibility, aggregation logic, and interpretation. Run all cells in order in Google Colab. The notebook has **not** been executed against the source CSV during revision; check the printed validation results before using the findings in your report.

## Code review: targeted improvements

| Original approach | Recommendation | Why it matters |
|---|---|---|
| Clone repository with a token embedded in the URL | Try a normal clone first; use Colab Secret `Git_Key` only for a private repository | Avoid exposing credentials in notebook output or error messages |
| Repeated country-level aggregations | Build one year-by-country sales table, then derive totals and growth | Reduces duplicated logic and inconsistent filters |
| `drop_duplicates()` across customer attributes | Check profile conflicts, then enforce one record per `CustomerKey` | A customer must not be counted twice because an attribute differs |
| Unmatched product names become `Accessories` | Assign `Unclassified` and inspect unmatched products | Prevents silent classification errors |
| Distinct-product average price and row-weighted price both shown without prioritization | Name the two estimands and choose the distinct-product mean for the question | Avoids confusing product-list price with transaction-weighted price |
| Compare homeownership means using a p-value only | Report group sizes, difference, Welch test and confidence interval | Makes the size and uncertainty of the association interpretable |

**Scope:** This is a refactor, not a change in research questions. Product categories remain *inferred* from names because the original analysis did not contain an authoritative category field.

## 0. Setup and data validation

In [ ]:
from pathlib import Path
import base64
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

REPO_URL = "https://github.com/xl4357/MSSP6070.git"
REPO_DIR = Path("/content/MSSP6070")
CSV_PATH = REPO_DIR / "Assignments" / "InternetSales.csv"

if not CSV_PATH.exists():
    # Public repositories require no credentials.
    result = subprocess.run(
        ["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        # For private repositories, use a Colab Secret named Git_Key.
        from google.colab import userdata
        token = userdata.get("Git_Key")
        if not token:
            raise RuntimeError("Clone failed. Check repository access and Git_Key secret.")
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        subprocess.run(
            ["git", "-c", f"http.extraheader=AUTHORIZATION: basic {auth}",
             "clone", "--quiet", REPO_URL, str(REPO_DIR)],
            check=True, capture_output=True
        )

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {CSV_PATH}")

sales = pd.read_csv(
    CSV_PATH,
    parse_dates=["BirthDate", "DateFirstPurchase", "OrderDate", "DueDate", "ShipDate"],
    encoding="latin1", low_memory=False
)

required = {
    "CustomerKey", "OrderDate", "EnglishCountryRegionName", "SalesAmount",
    "YearlyIncome", "Gender", "HouseOwnerFlag", "EnglishProductName",
    "ProductKey", "UnitPrice", "SalesOrderNumber"
}
missing = required.difference(sales.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

sales["OrderYear"] = sales["OrderDate"].dt.year
print("Rows and columns:", sales.shape)
print("Years:", sorted(sales["OrderYear"].dropna().unique().tolist()))
print("Missing values in analysis columns:")
print(sales[list(sorted(required))].isna().sum().loc[lambda x: x > 0])
assert sales["SalesAmount"].notna().all(), "Review missing sales amounts before aggregation."
assert sales["OrderDate"].notna().all(), "Review missing order dates before annual comparisons."

**Review note.** The original notebook used `index_col="ID"`, although no subsequent analysis relied on that index. Keeping the default row index makes the data-loading step simpler. The code also validates required columns before analysis. The repository is cloned only when the CSV is absent; restart the Colab runtime if you need a fresh copy.

## 1. Internet sales by year and country/region

In [ ]:
year_country_sales = (
    sales.groupby(["OrderYear", "EnglishCountryRegionName"])["SalesAmount"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

sales_millions = year_country_sales / 1_000_000
country_totals_millions = sales_millions.sum().sort_values(ascending=False)

display(sales_millions.round(3))
print("Total sales by country/region (USD millions):")
display(country_totals_millions.round(3).to_frame("Sales_USD_Millions"))

In [ ]:
ax = sales_millions.plot(marker="o", figsize=(11, 5))
ax.set(title="Internet Sales by Year and Country/Region",
       xlabel="Order Year", ylabel="Sales (USD millions)")
ax.set_xticks(sales_millions.index)
ax.grid(alpha=0.25)
ax.legend(title="Country/Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Year-over-year growth

**Review note.** The original code selected 2011–2013 without explaining the restriction. Those years were used as the full-year comparison window in the original report; 2010 and 2014 were treated as partial years. Verify this assumption against the order-date coverage before interpreting growth. The code below makes the comparison years explicit rather than silently dropping years.

In [ ]:
date_coverage = sales.groupby("OrderYear")["OrderDate"].agg(["min", "max", "count"])
display(date_coverage)

COMPARABLE_YEARS = [2011, 2012, 2013]  # Verify against the date coverage above.
if not set(COMPARABLE_YEARS).issubset(year_country_sales.index):
    raise ValueError("At least one comparison year is absent from the data.")

growth_pct = year_country_sales.loc[COMPARABLE_YEARS].pct_change() * 100
# Do not display the first year's undefined year-over-year growth.
growth_pct = growth_pct.iloc[1:]
display(growth_pct.round(2))

fastest_growth = pd.DataFrame({
    "FastestGrowingRegion": growth_pct.idxmax(axis=1),
    "GrowthPercent": growth_pct.max(axis=1)
})
display(fastest_growth.round(2))

In [ ]:
ax = growth_pct.T.plot.bar(figsize=(11, 5))
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Year-over-Year Sales Growth by Country/Region",
       xlabel="Country/Region", ylabel="Growth (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

**Interpretation caution.** A high growth percentage after a low-sales year can reflect a base effect. Compare absolute dollar changes as well as percentages before recommending market expansion.

In [ ]:
sales_change_millions = (
    year_country_sales.loc[COMPARABLE_YEARS].diff().iloc[1:] / 1_000_000
)
print("Year-over-year change in sales (USD millions):")
display(sales_change_millions.round(3))

## 2. Yearly income and gender by country/region

**Review note.** `drop_duplicates()` on all profile columns can leave multiple rows for one customer if the recorded attributes conflict. The code checks that risk and then creates one customer-level record per `CustomerKey`. Conflicting profiles should be investigated, not resolved arbitrarily.

In [ ]:
profile_cols = ["EnglishCountryRegionName", "YearlyIncome", "Gender", "HouseOwnerFlag"]
profile_variation = sales.groupby("CustomerKey")[profile_cols].nunique(dropna=False)
conflicts = profile_variation.gt(1).any(axis=1)
print("Customers with conflicting profile values:", int(conflicts.sum()))
if conflicts.any():
    display(profile_variation.loc[conflicts].head(10))
    raise ValueError("Conflicting customer profiles detected. Resolve before customer-level analysis.")

customers = sales[["CustomerKey", *profile_cols]].drop_duplicates(subset="CustomerKey").copy()
assert customers["CustomerKey"].is_unique
print("Unique customers:", len(customers))

In [ ]:
income_summary = (
    customers.groupby("EnglishCountryRegionName")["YearlyIncome"]
    .agg(Customers="count", Minimum="min", Maximum="max",
         Mean="mean", StandardDeviation="std")
)
display(income_summary.round(2))

gender_counts = pd.crosstab(customers["EnglishCountryRegionName"], customers["Gender"])
gender_percent = gender_counts.div(gender_counts.sum(axis=1), axis=0) * 100
print("Gender counts:")
display(gender_counts)
print("Gender distribution (% within country/region):")
display(gender_percent.round(2))

In [ ]:
ax = gender_percent.plot.bar(stacked=True, figsize=(10, 5))
ax.set(title="Gender Distribution by Country/Region",
       xlabel="Country/Region", ylabel="Customers (%)")
ax.legend(title="Gender", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

**Statistical note.** Gender is categorical: counts and percentages are more interpretable than a numerical standard deviation. The income standard deviation measures dispersion within each region; it does not establish differences in purchasing behavior.

## 3. Highest-sales product category and its average price

**Review note.** The CSV analysis uses product names to infer categories. The original `else: Accessories` rule silently labels every unmatched name as an accessory. This revision leaves unmatched names as `Unclassified` and prints them for inspection. For a final business report, an official product-category lookup is preferable if available.

In [ ]:
def classify_product(name):
    if pd.isna(name):
        return "Unclassified"
    name = str(name)
    if name.startswith(("Mountain-", "Road-", "Touring-")):
        return "Bikes"
    if any(term in name for term in ("Jersey", "Socks", "Gloves", "Vest", "Shorts")):
        return "Clothing"
    if any(term in name for term in ("Tire", "Tube")):
        return "Components"
    # These are heuristics, not an authoritative category mapping.
    if any(term in name for term in (
        "Helmet", "Bottle", "Cage", "Pump", "Lock", "Fender", "Rack",
        "Clean", "Patch", "Hydration", "Cap", "Stand"
    )):
        return "Accessories"
    return "Unclassified"

sales["ProductCategory"] = sales["EnglishProductName"].map(classify_product)
unknown_products = (
    sales.loc[sales["ProductCategory"].eq("Unclassified"),
              ["ProductKey", "EnglishProductName"]]
    .drop_duplicates().sort_values("EnglishProductName")
)
print("Unclassified distinct products:", len(unknown_products))
display(unknown_products.head(50))

**Decision required.** If the `Unclassified` table is nonempty, review those names and update the mapping before reporting final category totals. Do **not** automatically treat unclassified products as accessories. This may change the category-level results relative to the original notebook.

In [ ]:
category_summary = (
    sales.groupby("ProductCategory")
    .agg(TotalSales=("SalesAmount", "sum"),
         DistinctProducts=("ProductKey", "nunique"),
         SalesLines=("SalesAmount", "size"))
    .sort_values("TotalSales", ascending=False)
)
category_summary["SalesSharePercent"] = (
    100 * category_summary["TotalSales"] / category_summary["TotalSales"].sum()
)
display(category_summary.round(2))

# A distinct-product mean gives each ProductKey equal weight.
product_price_check = sales.groupby("ProductKey")["UnitPrice"].nunique(dropna=False)
if product_price_check.gt(1).any():
    print("Caution: Some ProductKeys have multiple recorded UnitPrice values.")
    print("The distinct-product mean below uses the first observed price per product.")

products = sales.drop_duplicates("ProductKey")[["ProductKey", "ProductCategory", "UnitPrice"]]
mean_listed_price = products.groupby("ProductCategory")["UnitPrice"].mean()
# A sales-line-weighted mean answers a different question.
mean_line_price = sales.groupby("ProductCategory")["UnitPrice"].mean()
price_comparison = pd.DataFrame({
    "MeanPricePerDistinctProduct": mean_listed_price,
    "MeanPricePerSalesLine": mean_line_price
})
display(price_comparison.round(2))

winner = category_summary.index[0]
print(f"Top category: {winner}")
print(f"Average distinct-product price: ${mean_listed_price.loc[winner]:,.2f}")

In [ ]:
ax = (category_summary["TotalSales"] / 1_000_000).plot.bar(figsize=(9, 5))
ax.set(title="Internet Sales by Inferred Product Category",
       xlabel="Category", ylabel="Sales (USD millions)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

**Interpretation caution.** `SalesLines` counts dataset rows, not necessarily unique orders. Product-level price is also not the same as realized revenue per order. Use `SalesOrderNumber.nunique()` when you need an order count.

## 4. Does homeownership make a difference in sales?

**Review note.** Compare customer-level spending, not individual sales lines. Each customer contributes one cumulative sales value. This avoids treating repeated purchases from the same customer as independent observations. The comparison remains observational and does not identify a causal effect.

In [ ]:
customer_sales = (
    sales.groupby("CustomerKey")
    .agg(TotalSales=("SalesAmount", "sum"),
         Orders=("SalesOrderNumber", "nunique"))
    .join(customers.set_index("CustomerKey")[["HouseOwnerFlag"]])
    .reset_index()
)

home_summary = (
    customer_sales.groupby("HouseOwnerFlag")
    .agg(Customers=("CustomerKey", "nunique"),
         AverageCustomerSales=("TotalSales", "mean"),
         MedianCustomerSales=("TotalSales", "median"),
         TotalSales=("TotalSales", "sum"),
         AverageOrders=("Orders", "mean"))
)
home_summary["CustomerSharePercent"] = 100 * home_summary["Customers"] / home_summary["Customers"].sum()
home_summary["SalesSharePercent"] = 100 * home_summary["TotalSales"] / home_summary["TotalSales"].sum()
display(home_summary.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
labels = {0: "Non-homeowners", 1: "Homeowners"}
values = home_summary["AverageCustomerSales"]
ax.bar([labels.get(k, str(k)) for k in values.index], values.to_numpy())
ax.set(title="Average Cumulative Sales per Customer by Homeownership",
       ylabel="Sales per Customer (USD)")
plt.tight_layout()
plt.show()

In [ ]:
homeowners = customer_sales.loc[customer_sales["HouseOwnerFlag"].eq(1), "TotalSales"].dropna()
nonhomeowners = customer_sales.loc[customer_sales["HouseOwnerFlag"].eq(0), "TotalSales"].dropna()
if min(len(homeowners), len(nonhomeowners)) < 2:
    raise ValueError("Both groups need at least two customers for Welch's t-test.")

result = stats.ttest_ind(homeowners, nonhomeowners, equal_var=False)
difference = homeowners.mean() - nonhomeowners.mean()
relative_difference_pct = 100 * difference / nonhomeowners.mean()

# Welch-Satterthwaite degrees of freedom and 95% confidence interval.
a = homeowners.var(ddof=1) / len(homeowners)
b = nonhomeowners.var(ddof=1) / len(nonhomeowners)
se = np.sqrt(a + b)
df = (a + b)**2 / (a**2 / (len(homeowners) - 1) + b**2 / (len(nonhomeowners) - 1))
ci_low, ci_high = stats.t.interval(0.95, df, loc=difference, scale=se)

print(f"Group sizes: homeowners={len(homeowners)}, non-homeowners={len(nonhomeowners)}")
print(f"Difference in mean cumulative customer sales: ${difference:,.2f}")
print(f"Relative difference: {relative_difference_pct:.2f}%")
print(f"Welch t-statistic: {result.statistic:.4f}")
print(f"Two-sided p-value: {result.pvalue:.6g}")
print(f"95% confidence interval for mean difference: (${ci_low:,.2f}, ${ci_high:,.2f})")

**Interpretation and limitations.** The Welch test allows unequal variances and compares mean *cumulative spending per observed customer*. It does not control for income, country/region, age, or length of time as a customer. A small p-value supports evidence of a group mean difference, **not** the claim that buying a home increases spending. The distribution of customer spending may be right-skewed; inspect medians and consider a sensitivity analysis before making a targeting recommendation.

## 5. Final validation checks

In [ ]:
# Revenue must be conserved across each aggregation.
raw_total = sales["SalesAmount"].sum()
assert np.isclose(year_country_sales.to_numpy().sum(), raw_total)
assert np.isclose(category_summary["TotalSales"].sum(), raw_total)
assert np.isclose(customer_sales["TotalSales"].sum(), raw_total)
assert len(customer_sales) == customers["CustomerKey"].nunique()

print("Validation passed: annual, category, and customer totals reconcile.")
print("Total revenue (USD):", round(raw_total, 2))
print("Unique customers:", customers["CustomerKey"].nunique())
print("Unclassified distinct products:", len(unknown_products))

## Reporting checklist

- Confirm that the 2011–2013 growth window contains comparable full years.
- Review all `Unclassified` product names; category sales and prices depend on this mapping.
- Report gender using counts and percentages; income using mean, min, max, and standard deviation.
- Distinguish the mean price of distinct products from the sales-line-weighted mean.
- State that homeownership results are **associations**, not causal estimates.
- Move figures to the report's Appendices and refer to them in the analysis text.

**Suggested next analytical improvement:** To evaluate whether the homeownership association persists after accounting for income and country, fit a customer-level regression with controls. Such a model would still be observational and would not by itself establish causality.